# 🪄 Rune Goblin — Fine-Tune MiniCPM-V 4.6 (Vision Spell Reader) on Modal

Fine-tune **`openbmb/MiniCPM-V-4.6`** to look at a hand-drawn RuneLang canvas + the game
state and emit `visual_reading` + `spell` JSON — the visual spell engine for Rune Goblin.

**Why ms-swift and not Unsloth?** The reference Colab uses Unsloth's `FastVisionModel`, but
Unsloth does **not** support the MiniCPM-V architecture. OpenBMB officially supports
**ms-swift** and **LLaMA-Factory** for MiniCPM-V 4.6, so this notebook uses **ms-swift**
(native `minicpmv4_6` support, handles the vision collator/template).

**Flow:** pull `ASHu2/rune_goblin_visual_dataset` → LoRA train (logging to W&B project
`goblin`) → eval → merge + push the model, adapter, and GGUF to `ASHu2/goblinV1`.

## How to run on Modal

You have ~$500 in Modal credits. Two ways to use them:

**A. Modal Notebook (what this file is built for) — simplest.**
1. **modal.com → Notebooks → New**, upload this `.ipynb`.
2. Attach a GPU. For this ~2.6B model, **A100-40GB** is the sweet spot (fastest, ~$0.50/run);
   **L4/A10 (24 GB)** also work and are slightly cheaper. Avoid T4 (no bf16).
3. Add two Modal **Secrets**: `huggingface-secret` (key `HF_TOKEN`) and `wandb-secret`
   (key `WANDB_API_KEY`). Locally they're read from `.env`; or paste into the Config cell.
4. **Run All.**

**B. Unattended job.** The **Appendix** at the end is a `modal run` script that does the same
thing headless (Volume cache + retries).

> Portable: also runs unchanged on Colab/RunPod/local — anywhere with an NVIDIA GPU + CUDA.

## 1 · Config

Edit these, then run top-to-bottom.

In [ ]:
import os
try:
    from dotenv import load_dotenv; load_dotenv()   # picks up HF_TOKEN / WANDB from a local .env
except Exception:
    pass

# --- model ---
MODEL_ID   = "openbmb/MiniCPM-V-4.6"     # the trainable safetensors base (NOT the GGUF)
MODEL_TYPE = "minicpmv4_6"               # ms-swift model_type for MiniCPM-V 4.6 (no hyphens!)

# --- auth (Modal: use Secrets; local: from .env; or paste here) ---
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Make ms-swift fetch the base model from Hugging Face (fast CDN, uses HF_TOKEN) instead of
# its default ModelScope Hub, which is slow (~700 kB/s) from US/Modal datacenters.
os.environ["USE_HF"] = "1"

# --- dataset (already uploaded) ---
HF_DATASET_REPO = os.environ.get("RG_DATASET_REPO", "ASHu2/rune_goblin_visual_dataset")
DATASET_DIR     = os.environ.get("RG_DATASET_DIR", "rune_goblin_visual_dataset")

# --- ONE Hub repo for everything: merged safetensors (root), LoRA (lora/), GGUF (gguf/) ---
MODEL_REPO = "ASHu2/goblinV1"
OUTPUT_DIR = "outputs/rune-goblin-vision-lora"

# --- experiment tracking (Weights & Biases) ---
WANDB_API_KEY = os.environ.get("WANDB") or os.environ.get("WANDB_API_KEY", "")
WANDB_PROJECT = "goblin"
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
REPORT_TO = "wandb" if WANDB_API_KEY else "none"

# --- hyperparameters (sensible defaults for the 5k set) ---
EPOCHS      = 3
LORA_RANK   = 16
LORA_ALPHA  = 32
LR          = 1e-4
BATCH       = 4          # per-device; lower to 1-2 on a 24GB card
GRAD_ACCUM  = 4          # effective batch = BATCH * GRAD_ACCUM
MAX_LEN     = 2048
FREEZE_VIT  = True       # train LLM+resampler only. False to also adapt the vision tower.

print("model :", MODEL_ID)
print("data  :", HF_DATASET_REPO)
print("output:", MODEL_REPO)
print("token :", bool(HF_TOKEN), "| wandb:", "on" if WANDB_API_KEY else "off")

## 2 · Install ms-swift + dependencies

`ms-swift` pulls a compatible `transformers`. SDPA attention avoids building flash-attn.

In [ ]:
%pip install -q -U "ms-swift>=3.0" "transformers>=4.49" "accelerate>=0.34" \
    peft timm pillow sentencepiece "trl<1.0" qwen-vl-utils wandb
# Optional (faster attention, needs a compile): %pip install -q flash-attn --no-build-isolation
import subprocess
print(subprocess.run(["swift", "--help"], capture_output=True, text=True).stdout[:200] or "swift installed")

## 3 · Get the dataset onto this machine

The dataset has 5,000 tiny images. Downloading them as **loose files is slow** (~4 min) because
HF fetches each one separately — you're latency-bound, not bandwidth-bound, and a token doesn't
help. So we download only the jsonl splits **+ a single `images.zip`** (one ~73 MB file → seconds)
and unzip it. `allow_patterns` skips the 5,000 loose files entirely.

(If you mounted the folder via a Modal Volume instead, set `RG_DATASET_DIR` and clear
`RG_DATASET_REPO`.)

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import snapshot_download

if HF_DATASET_REPO:
    # pull jsonl/metadata + the single images.zip; skip the 5000 loose images/ files
    DATASET_DIR = snapshot_download(
        repo_id=HF_DATASET_REPO, repo_type="dataset",
        local_dir="rg_visual_data", token=HF_TOKEN or None,
        allow_patterns=["*.jsonl", "*.json", "*.csv", "*.md", "images.zip"],
    )

root = Path(DATASET_DIR).resolve()
zip_path = root / "images.zip"
if zip_path.exists() and not (root / "images").exists():
    print("unzipping images.zip ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(root)

print("DATASET_DIR =", root)
need = ["train_messages.jsonl", "validation_messages.jsonl", "images"]
missing = [n for n in need if not (root / n).exists()]
assert not missing, f"missing in DATASET_DIR: {missing}. Set HF_DATASET_REPO or mount the folder."
print("images on disk:", len(list((root / "images").glob("*.jpg"))))

### (Optional) Re-upload the dataset to HF Hub

Already done for `ASHu2/rune_goblin_visual_dataset`. Use this only to refresh it — run it
**where the data folder exists** (your laptop), not on Modal. Flip `DO_UPLOAD = True`.

In [ ]:
DO_UPLOAD = False
if DO_UPLOAD:
    from huggingface_hub import create_repo, upload_folder
    repo = "ASHu2/rune_goblin_visual_dataset"
    local_folder = "data/rune_goblin_visual_dataset_5000/rune_goblin_visual_dataset"
    create_repo(repo, repo_type="dataset", private=False, exist_ok=True, token=HF_TOKEN)
    upload_folder(folder_path=local_folder, repo_id=repo, repo_type="dataset",
                  token=HF_TOKEN, ignore_patterns=["*:Zone.Identifier", "*.ipynb"])
    print("uploaded ->", repo)

## 4 · Convert the provided splits to ms-swift format

ms-swift wants `{"messages": [...], "images": ["/abs/path.jpg"]}`. We convert `*_messages.jsonl`
and rewrite image paths to absolute (verifying each exists). The `<image>` token already lives
in the user turn, which is what the template expects.

In [ ]:
import json, os

def to_swift(src, dst):
    kept, skipped = 0, 0
    with open(src) as f, open(dst, "w") as o:
        for line in f:
            r = json.loads(line)
            img = str((root / r["image"]).resolve())
            if not os.path.exists(img):   # source dataset references one missing image
                skipped += 1
                continue
            print(json.dumps({"messages": r["messages"], "images": [img]}, ensure_ascii=False), file=o)
            kept += 1
    return kept, skipped

TRAIN = "rg_swift_train.jsonl"
VAL   = "rg_swift_val.jsonl"
for split, dst in [("train_messages.jsonl", TRAIN), ("validation_messages.jsonl", VAL)]:
    kept, skipped = to_swift(root / split, dst)
    print(f"{split}: kept {kept}" + (f", skipped {skipped} (missing image)" if skipped else ""))

### Sanity check: look at one sample (image + target spell JSON)

In [ ]:
import json
from PIL import Image
from IPython.display import display

rec = json.loads(open("rg_swift_train.jsonl").readline())
user = next(m["content"] for m in rec["messages"] if m["role"] == "user")
asst = next(m["content"] for m in rec["messages"] if m["role"] == "assistant")
print("USER:\n", user, "\n")
print("TARGET JSON:\n", json.dumps(json.loads(asst), indent=2)[:700])
display(Image.open(rec["images"][0]).resize((256, 256)))

## 5 · LoRA fine-tune with ms-swift

Runs `swift sft` as a subprocess arg-list (no shell quoting). ~2.6B params → LoRA in bf16 fits
on 24GB+. Logs stream below and to **W&B project `goblin`** (run name = timestamped).

> The swift model_type is **`minicpmv4_6`** (no hyphens) — not the `minicpm-v-4_6` in OpenBMB's
> README. If it still errors, drop the flag (auto-detected from `--model`). Older swift (2.x)
> used `--sft_type lora` instead of `--train_type lora`.

In [ ]:
import os, subprocess
from datetime import datetime

# W&B in `swift sft` is configured via ENV + --report_to (NOT --wandb_project/--wandb_exp_name,
# which are Megatron-SWIFT flags). W&B auto-reads WANDB_PROJECT / WANDB_NAME from the env.
WANDB_EXP_NAME = "rune-goblin-v46-lora-" + datetime.now().strftime("%Y%m%d-%H%M%S")
if REPORT_TO == "wandb":
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    os.environ["WANDB_NAME"] = WANDB_EXP_NAME

args = [
    "swift", "sft",
    "--model", MODEL_ID,
    "--model_type", MODEL_TYPE,
    "--dataset", TRAIN,
    "--val_dataset", VAL,
    # train_type defaults to "lora" in ms-swift; --lora_rank/--lora_alpha configure it.
    # (ms-swift 4.x dropped the --train_type flag name.)
    "--lora_rank", str(LORA_RANK), "--lora_alpha", str(LORA_ALPHA),
    "--freeze_vit", str(FREEZE_VIT).lower(),
    "--torch_dtype", "bfloat16", "--attn_impl", "sdpa",
    "--num_train_epochs", str(EPOCHS),
    "--per_device_train_batch_size", str(BATCH),
    "--gradient_accumulation_steps", str(GRAD_ACCUM),
    "--learning_rate", str(LR),
    "--max_length", str(MAX_LEN),
    "--gradient_checkpointing", "true",
    "--warmup_ratio", "0.03",
    "--eval_steps", "200", "--save_steps", "200", "--logging_steps", "10",
    "--save_total_limit", "2",
    "--dataset_num_proc", "4",
    "--output_dir", OUTPUT_DIR,
    "--report_to", REPORT_TO,
]

print(" ".join(args), "\n")
ret = subprocess.run(args).returncode
print("\nswift sft exit code:", ret)

### Locate the trained adapter

In [ ]:
import glob, os

ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "checkpoint-*"), recursive=True),
               key=os.path.getmtime)
assert ckpts, f"no checkpoint found under {OUTPUT_DIR} — did training finish?"
ADAPTER_DIR = ckpts[-1]
print("ADAPTER_DIR =", ADAPTER_DIR)
print("contents:", os.listdir(ADAPTER_DIR)[:12])

## 6 · Merge LoRA, then evaluate on the validation split

A swift-trained LoRA can't be cleanly re-attached with `PeftModel` (module-path mismatch), so we
**merge** it into the base (swift maps the modules correctly) and load the merged model with the
transformers-native multimodal API — **MiniCPM-V 4.6 has no `.chat()`**; it uses
`AutoModelForImageTextToText` + `AutoProcessor` + `model.generate(..., downsample_mode=...)`.
Then we check (a) valid `visual_reading`+`spell` JSON and (b) rune-recognition recall vs the
ground truth in `*_full.jsonl`.

In [ ]:
import os, subprocess, torch
from transformers import AutoModelForImageTextToText, AutoProcessor

os.environ["USE_HF"] = "1"   # merge re-loads the base; pull from HF not ModelScope

# merge the LoRA into the base -> standalone model (reused by the push + GGUF cells)
MERGED_DIR = ADAPTER_DIR.rstrip("/") + "-merged"
if not os.path.exists(MERGED_DIR):
    subprocess.run(["swift", "export", "--adapters", ADAPTER_DIR, "--merge_lora", "true",
                    "--output_dir", MERGED_DIR], check=True)

model = AutoModelForImageTextToText.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True,
).eval()
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
print("loaded merged model + processor from", MERGED_DIR)

In [ ]:
import json

SYSTEM = ("You are Rune Goblin, a tiny vision spell engine. Read hand-drawn RuneLang glyphs "
          "from the image, infer the drawn runes, apply RuneLang combo rules and the game state, "
          "then output valid JSON only. The JSON must contain visual_reading and spell. "
          "Spells should be weird, funny, balanced, and game-safe.")

def extract_json(text):
    """Grab the first balanced {...} object from arbitrary model text."""
    if not text:
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None

def infer_one(image, user_text):
    # strip the literal <image> token; the processor injects the image via content parts
    user_text = user_text.replace("<image>\n", "").replace("<image>", "")
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": user_text}]},
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", downsample_mode="16x",
    ).to(model.device)
    gen = model.generate(**inputs, downsample_mode="16x", max_new_tokens=512)
    return processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]

In [ ]:
import json
from PIL import Image

# ground-truth runes come from the *_full.jsonl
full_by_id = {}
with open(root / "rune_goblin_visual_5000_full.jsonl") as f:
    for line in f:
        r = json.loads(line); full_by_id[r["id"]] = r

N = 50  # bump up for a fuller eval
with open(root / "validation_messages.jsonl") as f:
    val_rows = [json.loads(l) for l in f][:N]

valid = 0; rune_hits = 0; rune_total = 0; shown = 0
for r in val_rows:
    rid = r["id"]; img = Image.open((root / r["image"]).resolve()).convert("RGB")
    user = next(m["content"] for m in r["messages"] if m["role"] == "user")
    out = infer_one(img, user)
    obj = extract_json(out)
    if obj and "visual_reading" in obj and "spell" in obj:
        valid += 1
        pred = set(obj["visual_reading"].get("detected_runes", []))
        gt = set(full_by_id[rid]["runes_ground_truth"])
        if gt:
            rune_hits += len(pred & gt); rune_total += len(gt)
    if shown < 3:
        print("="*60); print("pred:", (out or "")[:300]); shown += 1

print(f"\nValid JSON rate : {valid}/{len(val_rows)} = {valid/len(val_rows):.1%}")
if rune_total:
    print(f"Rune recall     : {rune_hits}/{rune_total} = {rune_hits/rune_total:.1%}")

## 7 · Push everything to `ASHu2/goblinV1`

The merged model was produced in the eval step above (`MERGED_DIR`). We push to one repo:
**merged safetensors** → root, **LoRA adapter + training args** → `lora/`, **GGUF**
(next section) → `gguf/`. (If you skipped eval, this cell merges first.)

In [ ]:
import os, subprocess

MERGED_DIR = ADAPTER_DIR.rstrip("/") + "-merged"
if not os.path.exists(MERGED_DIR):   # eval cell normally already merged
    subprocess.run(["swift", "export", "--adapters", ADAPTER_DIR, "--merge_lora", "true",
                    "--output_dir", MERGED_DIR], check=True)
print("merged model at:", MERGED_DIR)

In [ ]:
import os, re
from huggingface_hub import create_repo, upload_folder

def fix_base_model_card(d):
    """swift writes README base_model as a local cache path; HF metadata validation rejects it."""
    p = os.path.join(d, "README.md")
    if not os.path.exists(p):
        return
    s = open(p, encoding="utf-8").read()
    s = re.sub(r"/root/\.cache/huggingface/hub/\S+", "openbmb/MiniCPM-V-4.6", s)
    open(p, "w", encoding="utf-8").write(s)

if MODEL_REPO:
    fix_base_model_card(MERGED_DIR)
    fix_base_model_card(ADAPTER_DIR)
    create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)          # model repo
    upload_folder(folder_path=MERGED_DIR, repo_id=MODEL_REPO, token=HF_TOKEN,
                  commit_message="merged MiniCPM-V 4.6 + RuneLang LoRA")
    upload_folder(folder_path=ADAPTER_DIR, repo_id=MODEL_REPO, path_in_repo="lora",
                  token=HF_TOKEN, commit_message="LoRA adapter + training metadata")
    print("pushed merged model + adapter -> ", f"https://huggingface.co/{MODEL_REPO}")
else:
    print("MODEL_REPO blank — skipping push.")

## 8 · Build GGUF (Q8_0 + Q4_K_M + mmproj) and upload to `ASHu2/goblinV1/gguf`

Fine-tuned analogues of the `MiniCPM-V-4.6-gguf` you downloaded, for fast CPU/edge serving via
llama.cpp / Ollama. We convert the LLM to f16, then quantize to **Q8_0** (quality) and
**Q4_K_M** (small/fast), best-effort export the **mmproj** vision projector, and upload all to
`MODEL_REPO/gguf/`.

Notes: `convert_hf_to_gguf.py` can't emit q4 directly, so we build `llama-quantize` (a quick
`cmake`). We also patch the merged model's `tokenizer_class` (`TokenizersBackend`, a
transformers-5 name the converter can't resolve) to `PreTrainedTokenizerFast`. MiniCPM-V GGUF
is finicky — if a step fails, your safetensors model on the Hub still serves fine via transformers.

In [ ]:
import os, json, glob, subprocess

LLAMA = "llama.cpp"
if not os.path.isdir(LLAMA):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
subprocess.run(f"pip install -q -r {LLAMA}/requirements.txt", shell=True, check=True)

# (1) fix tokenizer_class the converter can't resolve (loads fine from tokenizer.json)
tc_path = os.path.join(MERGED_DIR, "tokenizer_config.json")
tc = json.load(open(tc_path))
if tc.get("tokenizer_class") in ("TokenizersBackend", None):
    tc["tokenizer_class"] = "PreTrainedTokenizerFast"
    json.dump(tc, open(tc_path, "w"))
    print("patched tokenizer_class -> PreTrainedTokenizerFast")

F16 = "rune-goblin-v46-f16.gguf"
produced = []

# (2) convert LLM -> f16 GGUF
subprocess.run(["python", f"{LLAMA}/convert_hf_to_gguf.py", MERGED_DIR,
                "--outfile", F16, "--outtype", "f16"], check=True)

# (3) build llama-quantize (convert can't emit q4_k_m)
subprocess.run("apt-get -qq update && apt-get -qq install -y cmake build-essential", shell=True, check=False)
subprocess.run(f"cmake -S {LLAMA} -B {LLAMA}/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF >/dev/null", shell=True, check=True)
subprocess.run(f"cmake --build {LLAMA}/build --target llama-quantize -j", shell=True, check=True)
QUANT = (glob.glob(f"{LLAMA}/build/bin/llama-quantize") + glob.glob(f"{LLAMA}/build/llama-quantize"))[0]

# (4) quantize to Q8_0 (quality) and Q4_K_M (small)
for qtype, out in [("Q8_0", "rune-goblin-v46-Q8_0.gguf"),
                   ("Q4_K_M", "rune-goblin-v46-Q4_K_M.gguf")]:
    subprocess.run([QUANT, F16, out, qtype], check=True)
    produced.append(out)

# (5) vision projector -> mmproj GGUF (best-effort; varies by llama.cpp build)
MMPROJ = "rune-goblin-v46-mmproj-f16.gguf"
try:
    subprocess.run(["python", f"{LLAMA}/convert_hf_to_gguf.py", MERGED_DIR,
                    "--mmproj", "--outfile", MMPROJ], check=True)
    produced.append(MMPROJ)
except subprocess.CalledProcessError:
    print("mmproj export failed on this llama.cpp version — see llama.cpp mtmd/MiniCPM-V docs.")

print("produced:", produced)

# (6) upload to MODEL_REPO/gguf/
if MODEL_REPO and produced:
    from huggingface_hub import create_repo, upload_file
    create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)
    for f in produced:
        upload_file(path_or_fileobj=f, path_in_repo=f"gguf/{os.path.basename(f)}",
                    repo_id=MODEL_REPO, token=HF_TOKEN)
    print("uploaded GGUF -> ", f"https://huggingface.co/{MODEL_REPO}/tree/main/gguf")
else:
    print("MODEL_REPO blank — built locally but not pushed.")

# Serve:  llama-server -m rune-goblin-v46-Q4_K_M.gguf --mmproj rune-goblin-v46-mmproj-f16.gguf

## 9 · Cost & GPU guidance ($500 Modal credits)

MiniCPM-V 4.6 is ~2.6B params and canvases are 256×256, so LoRA is cheap and short.

| GPU | VRAM | $/hr | est. run | est. $/run |
|---|---|---|---|---|
| L4 | 24 GB | $0.80 | ~25-35 min | ~$0.40 |
| A10 | 24 GB | $1.10 | ~20-30 min | ~$0.45 |
| L40S | 48 GB | $1.95 | ~12-18 min | ~$0.50 |
| **A100-40GB** | 40 GB | $2.10 | ~10-15 min | **~$0.50** (best speed) |
| A100-80GB | 80 GB | $2.50 | — | only for full-FT / unfreeze ViT |

Avoid **T4** (no bf16). Because the job is short, a faster GPU often costs about the *same
total*. **A100-40GB** → ~$0.50/run → ~900 runs on $500. CPU ~8 cores, RAM ~32 GB is plenty
(Modal defaults are fine in a Notebook).

## Appendix · Run unattended as a `modal run` script

Save as `modal_finetune_minicpmv.py`, create the secrets, then launch:

```bash
modal secret create huggingface-secret HF_TOKEN=hf_xxxxxxxx
modal secret create wandb-secret WANDB_API_KEY=xxxxxxxx
modal run modal_finetune_minicpmv.py --epochs 3
```

Caches the HF model in a Volume, logs to W&B (`goblin`), trains LoRA, then merges and pushes
the model + adapter to `ASHu2/goblinV1`.

```python
import modal

app = modal.App("rune-goblin-vision-finetune")

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("ms-swift>=3.0", "transformers>=4.49", "accelerate>=0.34",
                 "peft", "timm", "pillow", "sentencepiece", "trl<1.0",
                 "qwen-vl-utils", "wandb", "huggingface_hub")
    .env({"HF_HOME": "/cache/hf"})
)
vol = modal.Volume.from_name("rune-goblin-vol", create_if_missing=True)

@app.function(
    image=image,
    gpu="A100-40GB", cpu=8.0, memory=32768,
    timeout=6 * 60 * 60,
    volumes={"/cache": vol},
    secrets=[modal.Secret.from_name("huggingface-secret"),   # -> HF_TOKEN
             modal.Secret.from_name("wandb-secret")],        # -> WANDB_API_KEY
    retries=modal.Retries(max_retries=2),
)
def finetune(epochs: int = 3,
             hf_dataset_repo: str = "ASHu2/rune_goblin_visual_dataset",
             model_repo: str = "ASHu2/goblinV1"):
    import os, json, time, glob, zipfile, subprocess
    from pathlib import Path
    from huggingface_hub import snapshot_download, create_repo, upload_folder

    os.environ["USE_HF"] = "1"   # pull base model from HF (fast), not ModelScope
    os.environ["WANDB_PROJECT"] = "goblin"
    os.environ["WANDB_NAME"] = f"rune-goblin-v46-{int(time.time())}"
    tok = os.environ["HF_TOKEN"]
    # fast: pull jsonl + single images.zip (not 5000 loose files), then unzip
    root = Path(snapshot_download(hf_dataset_repo, repo_type="dataset", local_dir="/cache/data",
                                  token=tok, allow_patterns=["*.jsonl", "images.zip"]))
    if (root / "images.zip").exists() and not (root / "images").exists():
        with zipfile.ZipFile(root / "images.zip") as z:
            z.extractall(root)

    # convert provided splits -> ms-swift jsonl (skip the one missing image)
    for split, dst in [("train_messages.jsonl", "/cache/train.jsonl"),
                       ("validation_messages.jsonl", "/cache/val.jsonl")]:
        with open(root / split) as f, open(dst, "w") as o:
            for line in f:
                r = json.loads(line)
                img = str((root / r["image"]).resolve())
                if not os.path.exists(img):
                    continue
                print(json.dumps({"messages": r["messages"], "images": [img]}), file=o)

    out = "/cache/rune-goblin-vision-lora"
    subprocess.run([
        "swift", "sft", "--model", "openbmb/MiniCPM-V-4.6", "--model_type", "minicpmv4_6",
        "--dataset", "/cache/train.jsonl", "--val_dataset", "/cache/val.jsonl",
        "--lora_rank", "16", "--lora_alpha", "32", "--freeze_vit", "true",
        "--torch_dtype", "bfloat16", "--attn_impl", "sdpa", "--num_train_epochs", str(epochs),
        "--per_device_train_batch_size", "4", "--gradient_accumulation_steps", "4",
        "--learning_rate", "1e-4", "--max_length", "2048", "--gradient_checkpointing", "true",
        "--save_steps", "200", "--eval_steps", "200", "--logging_steps", "10",
        "--report_to", "wandb",
        "--output_dir", out,
    ], check=True)
    vol.commit()

    # merge LoRA + push merged model (root) and adapter (lora/) to the single repo
    ckpt = sorted(glob.glob(f"{out}/**/checkpoint-*", recursive=True), key=os.path.getmtime)[-1]
    merged = ckpt + "-merged"
    subprocess.run(["swift", "export", "--adapters", ckpt, "--merge_lora", "true",
                    "--output_dir", merged], check=True)
    create_repo(model_repo, exist_ok=True, token=tok)
    upload_folder(folder_path=merged, repo_id=model_repo, token=tok)
    upload_folder(folder_path=ckpt, repo_id=model_repo, path_in_repo="lora", token=tok)
    print("pushed ->", f"https://huggingface.co/{model_repo}")

@app.local_entrypoint()
def main(epochs: int = 3):
    finetune.remote(epochs=epochs)
```